# Customer Segmentation

In [320]:
#Import packages
import pandas as pd
import numpy as np
import scipy   #Riyazi, statistik və elmi problemləri həll etmək üçün alətdir.
import matplotlib.pyplot as plt
import seaborn as sns

In [321]:
#Dataseti cagir
df = pd.read_csv('https://raw.githubusercontent.com/HumayDS/A15---24-Reqemsal-data-analitika/refs/heads/main/dataset%20rfm.csv')

In [322]:
df.head()

,CustomerID,PurchaseDate,TransactionAmount,ProductInformation,OrderID,Location
0,8814,2023-04-11,943.31,Product C,890075,Tokyo
1,2188,2023-04-11,463.70,Product A,176819,London
2,4608,2023-04-11,80.28,Product A,340062,New York
3,2559,2023-04-11,221.29,Product A,239145,London
4,9482,2023-04-11,739.56,Product A,194545,Paris


In [323]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CustomerID          1000 non-null   int64  
 1   PurchaseDate        1000 non-null   object 
 2   TransactionAmount   1000 non-null   float64
 3   ProductInformation  1000 non-null   object 
 4   OrderID             1000 non-null   int64  
 5   Location            1000 non-null   object 
dtypes: float64(1), int64(2), object(3)
memory usage: 47.0+ KB


In [324]:
df['PurchaseDate'] = pd.to_datetime(df['PurchaseDate'])

In [325]:
  #NA's check
df.isna().sum()

,0
CustomerID,0
PurchaseDate,0
TransactionAmount,0
ProductInformation,0
OrderID,0
Location,0


In [326]:
df['CustomerID'].nunique()

946

In [327]:
df['ProductInformation'] = df['ProductInformation'].astype('category')
df['Location'] = df['Location'].astype('category')

#RFM  - Recency -- Frequency --  Monetary

#Recency  — müştərinin son alışından bu günə qədər neçə gün keçdiyini göstərir.

In [328]:
# analysis_date = datasetdəki son alış tarixindən 1 gün sonrakı tarix
#Əgər 1 gün əlavə etməsək:
#Son gün alış edən müştərinin Recency-si 0 olar.
analysis_date = df['PurchaseDate'].max() + pd.Timedelta(days=1)

In [329]:
analysis_date

Timestamp('2023-06-11 00:00:00')

In [330]:
recency = df.groupby('CustomerID')['PurchaseDate'].max().reset_index()

In [331]:
recency

,CustomerID,PurchaseDate
0,1011,2023-05-08
1,1025,2023-05-20
2,1029,2023-06-10
3,1046,2023-04-28
4,1049,2023-05-28
...,...,...
941,9941,2023-04-29
942,9950,2023-05-03
943,9954,2023-05-29
944,9985,2023-04-14


In [332]:
recency['Recency'] = (analysis_date - recency['PurchaseDate']).dt.days

In [333]:
recency

,CustomerID,PurchaseDate,Recency
0,1011,2023-05-08,34
1,1025,2023-05-20,22
2,1029,2023-06-10,1
3,1046,2023-04-28,44
4,1049,2023-05-28,14
...,...,...,...
941,9941,2023-04-29,43
942,9950,2023-05-03,39
943,9954,2023-05-29,13
944,9985,2023-04-14,58


In [334]:
recency = recency[['CustomerID', 'Recency']]

In [335]:
recency

,CustomerID,Recency
0,1011,34
1,1025,22
2,1029,1
3,1046,44
4,1049,14
...,...,...
941,9941,43
942,9950,39
943,9954,13
944,9985,58


# Frequency — müştərinin müəyyən dövrdə neçə dəfə alış etdiyini göstərir.

In [336]:
frequency = df.groupby('CustomerID')['OrderID'].count().reset_index()

In [337]:
frequency

,CustomerID,OrderID
0,1011,2
1,1025,1
2,1029,1
3,1046,1
4,1049,1
...,...,...
941,9941,1
942,9950,1
943,9954,1
944,9985,1


In [338]:
#Rename edek
frequency = frequency.rename(columns = {'OrderID': 'Frequency'})

In [339]:
frequency

,CustomerID,Frequency
0,1011,2
1,1025,1
2,1029,1
3,1046,1
4,1049,1
...,...,...
941,9941,1
942,9950,1
943,9954,1
944,9985,1


Monetary - hər müştərinin ümumi xərci (total spend)

In [340]:
monetary = df.groupby('CustomerID')['TransactionAmount'].sum().reset_index()

monetary.columns = ['CustomerID', 'Monetary']

In [341]:
monetary

,CustomerID,Monetary
0,1011,1129.02
1,1025,359.29
2,1029,704.99
3,1046,859.82
4,1049,225.72
...,...,...
941,9941,960.53
942,9950,679.11
943,9954,798.01
944,9985,36.10


# RFM-  Bu addımda hər bir müştəri üçün recency, frequency və monetary göstəricilərini bir DataFrame-də birləşdiririk.

Bu, müştərinin davranışı haqqında ümumi və tam mənzərə əldə etməyə imkan verir.

In [342]:
rfm = recency.merge(frequency, on='CustomerID')
rfm = rfm.merge(monetary, on='CustomerID')

In [343]:
rfm

,CustomerID,Recency,Frequency,Monetary
0,1011,34,2,1129.02
1,1025,22,1,359.29
2,1029,1,1,704.99
3,1046,44,1,859.82
4,1049,14,1,225.72
...,...,...,...,...
941,9941,43,1,960.53
942,9950,39,1,679.11
943,9954,13,1,798.01
944,9985,58,1,36.10


In [344]:
rfm.describe().T

,count,mean,std,min,25%,50%,75%,max
CustomerID,946.0,5566.216702,2598.266606,1011.00,3281.75,5557.500,7824.750,9991.00
Recency,946.0,30.983087,17.318484,1.00,15.00,32.000,45.000,61.00
Frequency,946.0,1.057082,0.245418,1.00,1.00,1.000,1.000,3.00
Monetary,946.0,542.999799,324.382398,12.13,266.64,542.895,782.695,2379.45


In [345]:
rfm['Frequency'].value_counts().sort_index()

,count
Frequency,
1,895
2,48
3,3


##R, F, M üçün score ver (1–5)
#Recency az olduqca yaxşıdır (tərs scoring).

Recency göstəricisi üçün daha aşağı dəyər o deməkdir ki, müştəri son alışını xeyli əvvəl edib və bu, passivləşməyə başlayan müştəriləri müəyyən etməyə kömək edir. Bu cür müştərilər saxlanılması üçün yenidən aktivləşdirmə (re-engagement) kampaniyaları ilə hədəflənə bilər.

Digər tərəfdən, daha yüksək frequency və daha yüksək monetary dəyərlər daha sadiq və yüksək dəyərli müştəriləri göstərir.

In [346]:
rfm['R_score'] = pd.qcut(
    rfm['Recency'],
    5,
    labels=[5,4,3,2,1]
)

In [347]:
rfm['M_score'] = pd.qcut(
    rfm['Monetary'],
    5,
    labels=[1,2,3,4,5]
)

In [348]:
rfm['F_score'] = rfm['Frequency'].apply(
    lambda x: 5 if x in [2, 3] else 4
)

In [349]:
rfm

,CustomerID,Recency,Frequency,Monetary,R_score,M_score,F_score
0,1011,34,2,1129.02,3,5,5
1,1025,22,1,359.29,4,2,4
2,1029,1,1,704.99,5,4,4
3,1046,44,1,859.82,2,5,4
4,1049,14,1,225.72,4,2,4
...,...,...,...,...,...,...,...
941,9941,43,1,960.53,2,5,4
942,9950,39,1,679.11,2,4,4
943,9954,13,1,798.01,5,4,4
944,9985,58,1,36.10,1,1,4


In [350]:
#RFM score hesabla

In [351]:
rfm['RFM_segment'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

In [352]:
rfm

,CustomerID,Recency,Frequency,Monetary,R_score,M_score,F_score,RFM_segment
0,1011,34,2,1129.02,3,5,5,355
1,1025,22,1,359.29,4,2,4,442
2,1029,1,1,704.99,5,4,4,544
3,1046,44,1,859.82,2,5,4,245
4,1049,14,1,225.72,4,2,4,442
...,...,...,...,...,...,...,...,...
941,9941,43,1,960.53,2,5,4,245
942,9950,39,1,679.11,2,4,4,244
943,9954,13,1,798.01,5,4,4,544
944,9985,58,1,36.10,1,1,4,141


R_score: 1–5

M_score: 1–5

F_score: 4 və 5

Məntiq
 VIP Customers

R ≥ 4

M ≥ 4

F = 5

 High Value

R ≥ 3

M ≥ 4

 Medium Value

R ≥ 3

M ≤ 3

 Low Value

Qalanların hamısı (R_score ≤ 2 olan bütün müştərilər)

In [353]:
rfm['Customer_Group'] = 'Low Value'

# Medium
rfm.loc[
    (rfm['R_score'] >= 3) &
    (rfm['M_score'] <= 3),
    'Customer_Group'
] = 'Medium Value'

# High
rfm.loc[
    (rfm['R_score'] >= 3) &
    (rfm['M_score'] >= 4),
    'Customer_Group'
] = 'High Value'

# VIP
rfm.loc[
    (rfm['R_score'] >= 4) &
    (rfm['M_score'] >= 4) &
    (rfm['F_score'] == 5),
    'Customer_Group'
] = 'VIP Customers'

In [354]:
rfm

,CustomerID,Recency,Frequency,Monetary,R_score,M_score,F_score,RFM_segment,Customer_Group
0,1011,34,2,1129.02,3,5,5,355,VIP Customers
1,1025,22,1,359.29,4,2,4,442,Low Value
2,1029,1,1,704.99,5,4,4,544,Low Value
3,1046,44,1,859.82,2,5,4,245,High Value
4,1049,14,1,225.72,4,2,4,442,Low Value
...,...,...,...,...,...,...,...,...,...
941,9941,43,1,960.53,2,5,4,245,High Value
942,9950,39,1,679.11,2,4,4,244,High Value
943,9954,13,1,798.01,5,4,4,544,Low Value
944,9985,58,1,36.10,1,1,4,141,Medium Value


In [355]:
rfm.to_excel("RFM_Table.xlsx", index=False)

In [356]:
rfm['Customer_Group'].value_counts()

,count
Customer_Group,
Low Value,379
Medium Value,339
High Value,200
VIP Customers,28


VIP → ən aktiv və ən çox xərcləyən

High → yaxşı müştəri bazası

Medium → potensial var

Low → ya köhnədir(lost), ya da az xərcləyir

In [357]:
segment_dist = (
    rfm['Customer_Group']
        .value_counts()
        .reset_index()
)

segment_dist.columns = ['Customer_Group', 'Count']

segment_dist['Percentage'] = (
    segment_dist['Count'] /
    segment_dist['Count'].sum() * 100
).round(2)

segment_dist

,Customer_Group,Count,Percentage
0,Low Value,379,40.06
1,Medium Value,339,35.84
2,High Value,200,21.14
3,VIP Customers,28,2.96


In [358]:
segment_revenue = (
    rfm.groupby('Customer_Group')
       .agg(
           Customer_Count=('CustomerID', 'count'),
           Total_Revenue=('Monetary', 'sum')
       )
       .reset_index()
)

# Revenue faizi
segment_revenue['Revenue_%'] = (
    segment_revenue['Total_Revenue'] /
    segment_revenue['Total_Revenue'].sum() * 100
).round(2)

# Customer faizi
segment_revenue['Customer_%'] = (
    segment_revenue['Customer_Count'] /
    segment_revenue['Customer_Count'].sum() * 100
).round(2)

segment_revenue.sort_values(by='Revenue_%', ascending=False)

,Customer_Group,Customer_Count,Total_Revenue,Revenue_%,Customer_%
1,Low Value,379,201462.10,39.22,40.06
0,High Value,200,163514.13,31.83,21.14
2,Medium Value,339,114932.15,22.37,35.84
3,VIP Customers,28,33769.43,6.57,2.96


#Insight

# 1) Pareto qaydası işləmir

Normalda:

20% müştəri → 80% gəlir

Bizdə:

24% (High + VIP) → cəmi 38% gəlir

Bu göstərir ki:

Gəlir müştərilər arasında bərabər paylanıb, premium konsentrasiya yoxdur.

# 2)Low Value təhlükəli siqnaldır

40% müştəri

39% gəlir

Deməli:

Aşağı dəyərli müştəri belə xeyli pul gətirir.

Bu 2 şeyi göstərə bilər:

Segmentasiya sərhədləri çox yumşaqdır

Müştəri bazası ümumiyyətlə homogen strukturdadır

# 3) VIP çox zəifdir

3% müştəri

6.5% gəlir

Bu premium biznes üçün aşağı göstəricidir.

Normal güclü retail-də:

5–10% müştəri

20–40% gəlir gətirir

Bizdə isə premium layer formalaşmayıb.

Bu biznes:

✔ Stabildir
 Premiumlaşmayıb
 Güclü VIP layer yoxdur
 Retention sistemi zəifdir

Əsas problem:

Müştəri dəyərini artıran mexanizm yoxdur (upsell, loyalty, subscription və s.)

RFM nəticələri göstərir ki, gəlir müştərilər arasında balanslı paylanmışdır və yüksək konsentrasiyalı premium seqment mövcud deyil. Bu isə value maximization strategiyasının zəif olduğunu göstərir.

Low Value → 40% müştəri, 39% gəlir

High Value → 21% müştəri, 32% gəlir

VIP → 3% müştəri, 6.5% gəlir

Bu balanslı paylanmadır

- Yekun Rəy - Təklif 1


Aparılmış RFM analizi texniki olaraq düzgün qurulub və hesablama baxımından səhv görünmür. Lakin nəticələrin strukturu göstərir ki, segmentlər arasında kəskin dəyər fərqi formalaşmayıb və premium (VIP) təbəqə zəifdir. Gəlir paylanması da yüksək konsentrasiyalı deyil.

Bu isə iki ehtimal yaradır:

Mövcud scoring və quantile yanaşması biznesin real davranış modelini tam əks etdirmir.

Segment sərhədləri biznes məqsədlərinə uyğun kalibr edilməyib.

Xüsusilə:

Frequency faktiki differensiasiya etmir

VIP seqment gəlirin çox kiçik hissəsini təşkil edir

Pareto effekti müşahidə olunmur

Bu səbəbdən mövcud nəticələr analitik baxımdan qəbul edilə bilər, lakin strateji qərar vermək üçün kifayət qədər dərin deyil.

- Nəticə

Biznes ehtiyacları, müştəri davranışı və gəlir konsentrasiyası nəzərə alınaraq RFM modelinin yenidən kalibrasiyası və təkrar analizi məqsədəuyğun hesab olunur.

Bu təkrar analizdə:

Quantile əvəzinə biznes əsaslı threshold-lar

Frequency üçün daha real diferensiasiya

Monetary paylanmasının struktur analizi

VIP seqmentin dəyər konsentrasiyasının yoxlanması

tövsiyə olunur.

Bu, daha dəqiq segmentasiya və daha effektiv strateji qərar verməyə imkan yaradacaq.

# Təklif- rəy 2
Hazırkı struktur:
Stabil retail modelidir.

İstənilən hədəf:
Premium və daha yüksək marjlı biznes qurmaqdırsa —
segmentasiya sərtləşdirilməli və upsell strategiyası aktiv qurulmalıdır.